In [ ]:
!pip install lightgbm
!pip install -U ipykernel

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt

data_folder:هون الباث  اللي موجود فيه الcsv fileلكل مستهلك 

انا عندي مسميه MT

result_folder: هون ال باث اللي بدك تحط فيه نتائج المشروع 

علماً انه انا مسمي ملف المشروع Elcmodel

In [3]:

# 1. تحديد المسارات والمجلدات
data_folder = r"C:\Users\LEGION 5\Desktop\Elcmodel\MT"
result_folder = r"C:\Users\LEGION 5\Desktop\Elcmodel\reslt"

os.makedirs(result_folder, exist_ok=True)
plots_folder = os.path.join(result_folder, "comparison_plots_stable")
os.makedirs(plots_folder, exist_ok=True)

client_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]
print(f"[+] Found {len(client_files)} client files in '{data_folder}' folder.")


[+] Found 370 client files in 'C:\Users\LEGION 5\Desktop\Elcmodel\MT' folder.


In [ ]:

comparison_results = []

# إعداد التقسيم الزمني المتقاطع (5 طيات)
tscv = TimeSeriesSplit(n_splits=5)

for idx, file_name in enumerate(client_files, 1):
    file_path = os.path.join(data_folder, file_name)
    df = pd.read_csv(file_path)
    
    if 'target' not in df.columns or len(df) < 120:
        continue
        
    client_id = file_name.replace('.csv', '')
    
    X = df.drop(columns=['target'])
    y = df['target']
    
    # تهيئة القوائم لتخزين النتائج لكل نموذج
    xgb_train_rmse_list, xgb_test_rmse_list = [], []
    xgb_train_mae_list, xgb_test_mae_list = [], []
    xgb_train_r2_list, xgb_test_r2_list = [], []
    
    lgb_train_rmse_list, lgb_test_rmse_list = [], []
    lgb_train_mae_list, lgb_test_mae_list = [], []
    lgb_train_r2_list, lgb_test_r2_list = [], []
    
    # متغيرات لتخزين آخر Fold (لغرض رسم التوقعات الأخيرة بدقة)
    last_y_test = None
    last_xgb_preds = None
    last_lgb_preds = None

    # 2. تطبيق الـ Time Series Cross-Validation
    for train_index, test_index in tscv.split(X):
        X_train_full, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train_full, y_test = y.iloc[train_index], y.iloc[test_index]
        
        val_size = int(len(X_train_full) * 0.15)
        if val_size < 5:
            continue
            
        X_train = X_train_full.iloc[:-val_size]
        y_train = y_train_full.iloc[:-val_size]
        
        X_val = X_train_full.iloc[-val_size:]
        y_val = y_train_full.iloc[-val_size:]
        
        # ==========================================
        # 3. تدريب XGBoost 
        # ==========================================
        xgb_model = xgb.XGBRegressor(
            n_estimators=150, learning_rate=0.03, max_depth=3,
            min_child_weight=5, reg_alpha=1.0, reg_lambda=1.0,
            random_state=42, n_jobs=-1, early_stopping_rounds=20, 
            tree_method='hist',  
            device='cuda',       
        )
        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        xgb_train_preds = xgb_model.predict(X_train)
        xgb_test_preds = xgb_model.predict(X_test)
        
        xgb_train_rmse_list.append(np.sqrt(mean_squared_error(y_train, xgb_train_preds)))
        xgb_test_rmse_list.append(np.sqrt(mean_squared_error(y_test, xgb_test_preds)))
        xgb_train_mae_list.append(mean_absolute_error(y_train, xgb_train_preds))
        xgb_test_mae_list.append(mean_absolute_error(y_test, xgb_test_preds))
        xgb_train_r2_list.append(r2_score(y_train, xgb_train_preds))
        xgb_test_r2_list.append(r2_score(y_test, xgb_test_preds))
        
        # ==========================================
        # 4. تدريب LightGBM 
        # ==========================================
        lgb_model = lgb.LGBMRegressor(
            n_estimators=150, learning_rate=0.03, max_depth=3,
            num_leaves=15, min_child_samples=20, reg_alpha=1.0, reg_lambda=1.0,
            random_state=42, n_jobs=-1, verbose=-1,
        )
        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
        )
        
        lgb_train_preds = lgb_model.predict(X_train)
        lgb_test_preds = lgb_model.predict(X_test)
        
        lgb_train_rmse_list.append(np.sqrt(mean_squared_error(y_train, lgb_train_preds)))
        lgb_test_rmse_list.append(np.sqrt(mean_squared_error(y_test, lgb_test_preds)))
        lgb_train_mae_list.append(mean_absolute_error(y_train, lgb_train_preds))
        lgb_test_mae_list.append(mean_absolute_error(y_test, lgb_test_preds))
        lgb_train_r2_list.append(r2_score(y_train, lgb_train_preds))
        lgb_test_r2_list.append(r2_score(y_test, lgb_test_preds))

        # الاحتفاظ بنتائج الـ Fold الأخير للرسم
        last_y_test = y_test
        last_xgb_preds = xgb_test_preds
        last_lgb_preds = lgb_test_preds

    if len(xgb_test_rmse_list) == 0:
        continue

    # حساب المتوسطات لكل مقياس
    xgb_train_rmse = np.mean(xgb_train_rmse_list)
    xgb_test_rmse = np.mean(xgb_test_rmse_list)
    xgb_rmse_gap = abs(xgb_test_rmse - xgb_train_rmse)
    
    xgb_train_mae = np.mean(xgb_train_mae_list)
    xgb_test_mae = np.mean(xgb_test_mae_list)
    xgb_mae_gap = abs(xgb_test_mae - xgb_train_mae)  
    
    xgb_train_r2 = np.mean(xgb_train_r2_list)
    xgb_test_r2 = np.mean(xgb_test_r2_list)
    
    lgb_train_rmse = np.mean(lgb_train_rmse_list)
    lgb_test_rmse = np.mean(lgb_test_rmse_list)
    lgb_rmse_gap = abs(lgb_test_rmse - lgb_train_rmse)
    
    lgb_train_mae = np.mean(lgb_train_mae_list)
    lgb_test_mae = np.mean(lgb_test_mae_list)
    lgb_mae_gap = abs(lgb_test_mae - lgb_train_mae)  
    
    lgb_train_r2 = np.mean(lgb_train_r2_list)
    lgb_test_r2 = np.mean(lgb_test_r2_list)
    
    # 5. المقارنة الهندسية
    stable_model = 'XGBoost' if xgb_rmse_gap < lgb_rmse_gap else 'LightGBM'
    best_performance_model = 'LightGBM' if lgb_test_rmse < xgb_test_rmse else 'XGBoost'
    
    comparison_results.append({
        'Client': file_name,
        'XGB_Train_RMSE': xgb_train_rmse, 'XGB_Test_RMSE': xgb_test_rmse, 'XGB_RMSE_Gap': xgb_rmse_gap,
        'XGB_Train_MAE': xgb_train_mae, 'XGB_Test_MAE': xgb_test_mae, 'XGB_MAE_Gap': xgb_mae_gap,
        'XGB_Train_R2': xgb_train_r2, 'XGB_Test_R2': xgb_test_r2,
        
        'LGB_Train_RMSE': lgb_train_rmse, 'LGB_Test_RMSE': lgb_test_rmse, 'LGB_RMSE_Gap': lgb_rmse_gap,
        'LGB_Train_MAE': lgb_train_mae, 'LGB_Test_MAE': lgb_test_mae, 'LGB_MAE_Gap': lgb_mae_gap,
        'LGB_Train_R2': lgb_train_r2, 'LGB_Test_R2': lgb_test_r2,
        
        'Most_Stable_Model': stable_model,
        'Best_Performance_Model': best_performance_model
    })

    # ==========================================
    # 6. رسم وتصدير الرسوم البيانية لكل عميل حصرياً هنا
    # ==========================================
    # أ. رسم وتصدير توقعات XGBoost لكل مستهلك
    plt.figure(figsize=(14, 6))
    plt.plot(last_y_test.to_numpy(), label="Actual Load (الحقيقي)", color="blue", linewidth=2)
    plt.plot(last_xgb_preds, label="Predicted Load (XGBoost)", color="red", linestyle="--", linewidth=2)

    plt.title(f"Electricity Load Forecast - XGBoost - {client_id}", fontsize=14)
    plt.xlabel("Test time steps (ساعات الاختبار)")
    plt.ylabel("Consumption (الاستهلاك)")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.7)
    # إنشاء مجلدات الرسم إذا لم تكن موجودة
    os.makedirs(plots_folder, exist_ok=True)
    plots_folder_xgb = os.path.join(plots_folder, "xgb")
    plots_folder_lgb = os.path.join(plots_folder, "lgb")
    os.makedirs(plots_folder_xgb, exist_ok=True)
    os.makedirs(plots_folder_lgb, exist_ok=True)

    plt.tight_layout()
    plot_path_xgb = os.path.join(plots_folder_xgb, f"{client_id}_xgb_plot.png")
    plt.savefig(plot_path_xgb, dpi=300, bbox_inches="tight")
    plt.close() # إغلاق الصورة لتفريغ الذاكرة

    # ب. رسم وتصدير توقعات LightGBM لكل مستهلك
    plt.figure(figsize=(14, 6))
    plt.plot(last_y_test.to_numpy(), label="Actual Load (الحقيقي)", color="blue", linewidth=2)
    plt.plot(last_lgb_preds, label="Predicted Load (LightGBM)", color="green", linestyle="--", linewidth=2)

    plt.title(f"Electricity Load Forecast - LightGBM - {client_id}", fontsize=14)
    plt.xlabel("Test time steps (ساعات الاختبار)")
    plt.ylabel("Consumption (الاستهلاك)")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.7)

    plot_path_lgb = os.path.join(plots_folder_lgb, f"{client_id}_lgb_plot.png")
    plt.savefig(plot_path_lgb, dpi=300, bbox_inches="tight")
    plt.close() # إغلاق الصورة لتفريغ الذاكرة

    print(f"[{idx}/{len(client_files)}] {client_id} -> Processed & Plots Saved Successfully!")

# 7. حفظ النتائج النهائية للمقارنة
if len(comparison_results) > 0:
    summary_df = pd.DataFrame(comparison_results)
    output_path = os.path.join(result_folder, "models_stability_comparison_tscv.csv")
    try:
        summary_df.to_csv(output_path, index=False)
    except PermissionError:
        output_path = os.path.join(
            result_folder,
            f"models_stability_comparison_tscv_{pd.Timestamp.now():%Y%m%d_%H%M%S}.csv"
        )
        summary_df.to_csv(output_path, index=False)
    print(f"\n--- Stability Analysis Saved Successfully at: {output_path} ---")
else:
    print("\n[!] No models were processed.")

In [9]:
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
import numpy as np
if 'comparison_results' not in globals():
    result_folder = r"C:\Users\LEGION 5\Desktop\Elcmodel\reslt"
    os.makedirs(result_folder, exist_ok=True)

    plots_folder = os.path.join(result_folder, "comparison_plots_stable")
    os.makedirs(plots_folder, exist_ok=True)

    results_path = os.path.join(
        result_folder, "models_stability_comparison_tscv_20261030_151919.csv"
    )
    if not os.path.exists(results_path):
        raise FileNotFoundError(
            "Results file not found. Run cell 3 first to generate comparison_results."
        )
    comparison_results = pd.read_csv(results_path).to_dict('records')

for res in comparison_results:
    client_name = str(res['Client']).replace('.csv', '')

    models = ['XGBoost (RMSE)', 'LightGBM (RMSE)', 'XGBoost (MAE)', 'LightGBM (MAE)']
    train_vals = [
        res['XGB_Train_RMSE'], res['LGB_Train_RMSE'],
        res['XGB_Train_MAE'], res['LGB_Train_MAE']
    ]
    test_vals = [
        res['XGB_Test_RMSE'], res['LGB_Test_RMSE'],
        res['XGB_Test_MAE'], res['LGB_Test_MAE']
    ]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, train_vals, width, label='Train', color='skyblue')
    ax.bar(x + width / 2, test_vals, width, label='Test', color='salmon')

    ax.set_ylabel('Error Value')
    ax.set_title(f'Train vs Test Errors for Client: {client_name}')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15)
    ax.legend()

    plt.tight_layout()
    plot_path = os.path.join(plots_folder, f"{client_name}_error_comparison.png")
    plt.savefig(plot_path)
    plt.close()

In [10]:
import pandas as pd
import os

output_path = os.path.join(result_folder, "models_stability_comparison_tscv.csv")

if os.path.exists(output_path):
    df_results = pd.read_csv(output_path)
    
    print("--- متوسط مقاييس الخطأ MAE لـ XGBoost ---")
    print(f"Train MAE: {df_results['XGB_Train_MAE'].mean():.3f}")
    print(f"Test MAE:  {df_results['XGB_Test_MAE'].mean():.3f}")
    print(f"MAE Gap:   {df_results['XGB_MAE_Gap'].mean():.3f}\n")
    
    print("--- متوسط مقاييس الخطأ MAE لـ LightGBM ---")
    print(f"Train MAE: {df_results['LGB_Train_MAE'].mean():.3f}")
    print(f"Test MAE:  {df_results['LGB_Test_MAE'].mean():.3f}")
    print(f"MAE Gap:   {df_results['LGB_MAE_Gap'].mean():.3f}")
else:
    print("ملف النتائج غير موجود.")

--- متوسط مقاييس الخطأ MAE لـ XGBoost ---
Train MAE: 29.128
Test MAE:  38.530
MAE Gap:   9.428

--- متوسط مقاييس الخطأ MAE لـ LightGBM ---
Train MAE: 29.172
Test MAE:  38.206
MAE Gap:   9.060
